1단계: PDF 텍스트 추출
- 라이브러리 설치: !pip install pdfplumber

In [1]:
# PDF 텍스트 추출 라이브러리 import
import pdfplumber

# PDF 파일 열기
with pdfplumber.open(path_or_fp="sample_korean_long.pdf") as pdf:
    text = ""
    for page in pdf.pages:
        text += page.extract_text() + "\n"

# 텍스트 추출 확인
print("총 텍스트 길이:", len(text))
print(text[:500]) # 앞부분만 확인

총 텍스트 길이: 906
제목: 인공지능 개요
인공지능은 현대 사회에서 다양한 분야에 적용되고 있습니다.
의료, 금융, 제조업, 교육 등 거의 모든 산업에서 인공지능 기술이 활용되고 있으며,
특히 대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여
사람과 유사한 방식으로 질문에 답변할 수 있습니다.
이러한 모델은 단순한 질의응답을 넘어, 문서 요약, 번역, 창의적 글쓰기 등
다양한 작업을 수행할 수 있어 실무에서 큰 가치를 제공합니다.
제목: 벡터 데이터베이스
벡터 데이터베이스는 인공지능과 함께 사용될 때 강력한 도구가 됩니다.
문서를 벡터로 변환하여 저장하면, 질문과 가장 유사한 문서를 빠르게 검색할 수 있습니다.
이는 단순 키워드 검색보다 훨씬 정교한 결과를 제공합니다.
Qdrant, Pinecone, Weaviate 등 다양한 벡터 데이터베이스가 존재하며,
특히 Qdrant는 고성능 검색과 인덱싱을 지원하여 RAG 시스템 구축에 널리 활용됩니다.
제목: RAG 시스템
RAG(Retrieval-


2단계: 텍스트 청크 분할
- 문단 단위 대신 토큰/문자 기준으로 일정 길이로 나누어 문맥 손실을 최소화
- overlap을 주어 청크 사이 문맥을 이어준다

In [2]:
# transformers AutoTokenizer 라이브러리 import
from transformers import AutoTokenizer

# AutoTokenizer 모델: 토큰 개수는 모델의 토큰 사전(vocabulary)과 토큰화 규칙에 따라 달라짐
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 토큰 함수 생성
def chunk_text_by_tokens(text, max_tokens=256, overlap=50):
    # 텍스트를 토큰 ID 리스트로 변환
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    # 두번째 루프 전에 206(256-50) step으로 증가 적용, 0번째 0:256, 1번째 206:462
    for i in range(0, len(tokens), max_tokens - overlap):
        chunk = tokens[i:i + max_tokens] # 토큰 범위 지정
        chunks.append(tokenizer.decode(chunk)) # 토큰을 다시 문자열로 변환
    return chunks

chunks = chunk_text_by_tokens(text, max_tokens=256, overlap=50)
print(len(chunks), "개의 청크 생성")

# 토큰 개수 확인
tokens = tokenizer.encode(text, add_special_tokens=False)
print("토큰 개수:", len(tokens))

2 개의 청크 생성
토큰 개수: 399


3단계: 임베딩 단계
- SentenceTransformer 같은 모델을 사용해 각 청크를 고차원 벡터(embedding)로 변환
- 이렇게 얻은 벡터는 의미적 유사성을 계산할 수 있어서, 검색(RAG), 분류, 추천 등에 활용

In [3]:
# SentenceTransformer 임베딩 라이브러리 import
from sentence_transformers import SentenceTransformer

# 다국어 지원 SentenceTransformer 모델 로드
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 미리 만든 청크 리스트를 임베딩으로 변환
embeddings = model.encode(chunks) # numpy.ndarray 형태의 베터 리스트, 각 벡터는 보통 384 차원

# 결과 확인
print("임베딩 벡터 개수:", len(embeddings)) # 청크 개수와 동일
print("첫번째 청크 임베딩 벡터 크기:", len(embeddings[0])) # 일반적으로 384 차원

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


임베딩 벡터 개수: 2
첫번째 청크 임베딩 벡터 크기: 384


4단계: Qdrant 컬렉션 생성
- 라이브러리 설치: !pip install qdrant-client

In [5]:
from qdrant_client import QdrantClient # Qdrant 클라이언트 import
from qdrant_client.http import models # Qdrant 클라이언트에서 서버와 통신할때 타입 정의 벡터 파라미터, PointStruct 구조체(id/vector/payload) 설정 모델 불러오는 역할 import

# Qdrant 서버 연결
client = QdrantClient(host="localhost", port=6333)

collection_name = "documents_pdf"

# 컬렉션 존재 여부 확인
try:
    client.get_collection(collection_name=collection_name)
    print(f" '${collection_name}' 컬렉션이 이미 존재합니다.")
except Exception:
    # 컬렉션이 없으면 새로 생성
    client.create_collection(
        collection_name=collection_name,
        # models.VectorParams(size 벡터 크기, distance 거리 계산 방식 정의)
        vectors_config=models.VectorParams(size=len(embeddings[0]), distance=models.Distance.COSINE),
    )
    print(f" '${collection_name} 컬렉션이 새로 생성되었습니다.")

 '$documents_pdf' 컬렉션이 이미 존재합니다.


5단계: Qdrant 인덱싱
- Upsert 인덱싱 작업

In [6]:
# points 리스트 생성
points = [
    # Qdrant에 저장할 데이터 구조체
    models.PointStruct(
        id=i, # 각 벡터의 고유 식별자(여기서는 인덱스 번호 i)
        vector=embeddings[i], # 실제 임베딩 벡터
        payload={"text": chunks[i]} # 벡터와 함께 저장할 메타데이터(여기서는 원문 청크 텍스트)
    )
    # 리스트 컴프리핸션을 사용해 embeddings 개수 만큼 반복 -> 모든 청크를 벡터와 함께 구조화
    for i in range(len(embeddings))
]
# upsert(insert+update): 새 데이터를 추가하거나 기존 데이터를 덮어씀
client.upsert(collection_name=collection_name, points=points) # 컬렉션에 저장, PointStruct 리스트를 전달->Qdrant DB 벡터와 텍스트 저장

print("인덱싱 완료")

인덱싱 완료


6단계: Qdrant 검색
- 검색 결과

In [8]:
# 질의문
query = "벡터 데이터베이스란 무엇인가?"

# 질의문 벡터 변환: SentenceTransformers 모델을 사용하여 질의문을 임베딩 벡터로 변환, 넘파이 배열을 파이썬 리스트로 반환(Qdrant 클라이언트는 리스트 타입을 요구함)
query_vector = model.encode(sentences=query).tolist() # 질의문 -> 벡터 변환(타입은 리스트)

# Qdrant 검색: query_points 호출시 Qdrant 서버가 검색 결과 개체를 반환
results = client.query_points( # query_points 메서드로 Qdrant DB에서 검색 실행
    collection_name=collection_name, # 컬렉션명
    query=query_vector, # 질의문 벡터
    limit=3 # 가장 유사한 상위 3건 결과 반환
)

# 결과 확인: 검색결과 객체 안에는 여러 속성이 있으며, 그중 핵심이 바로 points 이다
for r in results.points: # 검새된 포인트(PointStruct)들의 리스트이다
    # r.payload("text") 저장된 메타데이터(청크 텍스트), r.score 질의문 벡터와 해당 청크 벡터의 유사도 점수 출력
    print(r.payload["text"], "=> score", r.score)

제목: 인공지능 개요 인공지능은 현대 사회에서 다양한 분야에 적용되고 있습니다. 의료, 금융, 제조업, 교육 등 거의 모든 산업에서 인공지능 기술이 활용되고 있으며, 특히 대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여 사람과 유사한 방식으로 질문에 답변할 수 있습니다. 이러한 모델은 단순한 질의응답을 넘어, 문서 요약, 번역, 창의적 글쓰기 등 다양한 작업을 수행할 수 있어 실무에서 큰 가치를 제공합니다. 제목: 벡터 데이터베이스 벡터 데이터베이스는 인공지능과 함께 사용될 때 강력한 도구가 됩니다. 문서를 벡터로 변환하여 저장하면, 질문과 가장 유사한 문서를 빠르게 검색할 수 있습니다. 이는 단순 키워드 검색보다 훨씬 정교한 결과를 제공합니다. Qdrant, Pinecone, Weaviate 등 다양한 벡터 데이터베이스가 존재하며, 특히 Qdrant는 고성능 검색과 인덱싱을 지원하여 RAG 시스템 구축에 널리 활용됩니다. 제목: RAG 시스템 RAG(Retrieval-Augmented Generation)는 검색과 생성 모델을 결합한 방식입니다. 검색된 문서를 기반으로 LLM이 답변을 생성하기 때문에, 정확 => score 0.3993595
널리 활용됩니다. 제목: RAG 시스템 RAG(Retrieval-Augmented Generation)는 검색과 생성 모델을 결합한 방식입니다. 검색된 문서를 기반으로 LLM이 답변을 생성하기 때문에, 정확성과 신뢰성을 크게 향상시킬 수 있습니다. 예를 들어, 고객 지원 시스템에서 RAG를 활용하면 사용자의 질문과 가장 관련 있는 문서를 검색한 뒤, 그 내용을 바탕으로 정확한 답변을 생성할 수 있습니다. 제목: 실무 적용 사례 1. 의료 분야: 환자의 기록을 벡터화하여 유사 사례를 검색하고 진단 지원에 활용 2. 금융 분야: 투자 보고서를 검색해 리스크 분석에 활용 3. 교육 분야: 학생 질문에 맞는 학습 자료를 검색해 맞춤형 학습 제공 4. 제조업: 기술 문서를 검색해 문제 해결 속도를 향상 이처럼 벡터 검